In [0]:
%run ./cid_mapping_common_business

In [0]:
%run ../00_common/calc_ctable

In [0]:
# generate json data
ts_format = "yyyy-MM-dd'T'HH:mm:ss"

In [0]:
def calc_t_transaction_master_dataset(task_id):
    master_df = spark.table(f"{get_env_config('golden_consumer_master_database')}.t_transaction_master")

    select_cid_df = (master_df
                     .filter(F.col("task_id") == task_id)
                     .select(F.col("tran_mapping_consumer_id"))
                     .distinct()
    )

    merge_data_df = select_cid_df.alias("scd").join(master_df.alias("md"), ["tran_mapping_consumer_id"], "inner").select("md.*")

    result_df = (merge_data_df
        .withColumn("document_uuid", F.expr("uuid()"))
        .withColumn("record_uuid", F.expr("uuid()"))
        .withColumn("current_ts",F.date_format(F.current_timestamp(), ts_format))
        .withColumn("json_str",
            build_cid_mapping_json_udf(
                F.col("tran_action"),
                F.col("tran_srcs_code"),     # MasterConsumer.@TypeCode
                F.col("tran_srcs_code"),     # MappingConsumer.@Code
                F.col("tran_mrkt_code"),
                F.col("tran_brnd_code"),
                F.col("tran_order_id"),
                F.col("tran_mapping_consumer_id"),
                F.col("current_ts"),          # DocumentTimestamp
                F.col("document_uuid"),       # DocumentUUID
                F.col("record_uuid"),         # @RecordUUID
                F.col("current_ts"),          # MappingTimestamp
            )
        )
        .select(
            F.col("tran_id"),
            F.col("tran_mrkt_code"),
            F.col("tran_brnd_code"),
            F.col("tran_srcs_code"),
            F.col("tran_order_id"),
            F.col("json_str"),
            F.lit(task_id).alias("task_id"),
            F.current_timestamp().alias("creation_dt"),
            F.current_timestamp().alias("update_dt"),
        )
    )

    # cache
    result_df.cache()
    result_count = result_df.count()
    print(f"Total records to merge: {result_count}")
    
    if result_count > 0:
        base_table_name = f"{get_env_config('golden_consumer_combine_database')}.t_transaction_master_dataset"
        # merge
        base_table = DeltaTable.forName(spark, base_table_name)
        (
            base_table.alias("b")
            .merge(result_df.alias("u"),  
                f""" b.tran_mrkt_code = u.tran_mrkt_code and  
                        b.tran_id = u.tran_id and  
                        b.tran_brnd_code = u.tran_brnd_code and  
                        b.tran_srcs_code = u.tran_srcs_code and  
                        b.tran_order_id = u.tran_order_id 
                    """
            )
            .whenMatchedUpdate(
                set = {
                    "json_str": F.col("u.json_str"),
                    "task_id": F.col("u.task_id"),
                    "update_dt": F.col("u.update_dt")
                }
            )
            .whenNotMatchedInsertAll()
            .execute()
        )
        
        # calc ctable
        calc_ctable(base_table_name, None)

    # unpersist
    result_df.unpersist()

In [0]:
task_id = dbutils.widgets.get("task_id")
print(f"task_id: {task_id}")


with StepLogger("3.3_generate_trans_dataset", "03-3", "consumerlist", task_id=task_id) as logger:
    calc_t_transaction_master_dataset(task_id)